In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
import random
import tensorflow as tf

# Configure plotting
plt.style.use('default')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the creditcard dataset
df = pd.read_csv('d:/downloads/creditcard.csv/creditcard.csv')

# Print dataset info
print("Dataset Info:")
df.info()

# Show first few rows
print("\nFirst few rows of the dataset:")
print(df.head())

In [ ]:
df.sample(5)

In [ ]:
df.isnull().sum()

In [ ]:
# Get basic statistics for Amount and Time
print("Transaction Amount Statistics:")
print(df['Amount'].describe())
print("\nTransaction Time Statistics:")
print(df['Time'].describe())

In [ ]:
# Optimize memory usage by downcasting numerical columns
for col in df.columns:
    if df[col].dtype == 'float64':
        df[col] = pd.to_numeric(df[col], downcast='float')
    elif df[col].dtype == 'int64':
        df[col] = pd.to_numeric(df[col], downcast='integer')

In [ ]:
# Check duplicate values
df.duplicated().sum()

In [ ]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8,6)

In [ ]:
# Analyze time distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Time', hue='Class', bins=50)
plt.title('Distribution of Transactions over Time by Class')
plt.xlabel('Time (seconds)')
plt.ylabel('Count')

In [ ]:
# Plot class distribution
ax = sns.countplot(x='Class', data=df, palette='PuBu')
for container in ax.containers:
    ax.bar_label(container)
plt.title('Distribution of Normal vs Fraudulent Transactions')
plt.ylabel('Number of transactions')
plt.xlabel('Class (0: Normal, 1: Fraud)')

In [ ]:
# Plot amount distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Amount', hue='Class', bins=50)
plt.title('Distribution of Transaction Amounts by Class')
plt.xlabel('Amount')
plt.ylabel('Count')

In [ ]:
# Calculate correlation matrix
cols_to_correlate = ['Time', 'Amount', 'Class'] + [f'V{i}' for i in range(1, 29)]
corr_matrix = df[cols_to_correlate].corr('spearman')

# Create correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap='PuBu', center=0,
            mask=np.triu(np.ones_like(corr_matrix, dtype=bool)),
            fmt='.2f', square=True, cbar_kws={'shrink': .5})
plt.title('Feature Correlations')
plt.tight_layout()

In [ ]:
# Import StandardScaler
from sklearn.preprocessing import StandardScaler

# Scale Amount and Time features
scaler = StandardScaler()
df[['Amount', 'Time']] = scaler.fit_transform(df[['Amount', 'Time']])

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, roc_curve, auc, ConfusionMatrixDisplay

seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

X = df.copy()
X.drop(['Class'], axis=1, inplace=True)  # 'Class' is our target variable
y = df['Class']  # 1 for fraud, 0 for normal

# Stratified train-test split
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
for train_idx, test_idx in skfold.split(X,y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

sc = StandardScaler()
scaled_train = sc.fit_transform(X_train)
scaled_test = sc.transform(X_test)
X_train = pd.DataFrame(scaled_train, index=X_train.index, columns=X_train.columns)
X_test = pd.DataFrame(scaled_test, index=X_test.index, columns=X_test.columns)

X_train, y_train = RandomUnderSampler(sampling_strategy='majority').fit_resample(X_train, y_train)

In [ ]:
def model_comparison_evaluate(classifiers, X, y):
    print('K-Fold Cross-Validation:\n')
    for name, model in classifiers.items():
        print('{}:'.format(name))
        
        scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        
        for score in scoring:
            scores = cross_val_score(model, X, y, scoring=score, cv=skfold, n_jobs=-1)
            print('Mean {} score: {:.3f} ({:.3f})'.format(score, scores.mean(), scores.std()))
            
        print('\n')

In [ ]:
classifiers = { 'Random Forest Classifier':RandomForestClassifier(class_weight='balanced', random_state=seed),
                'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=seed)
              }

In [ ]:
model_comparison_evaluate(classifiers, X_train, y_train)

In [ ]:
model = RandomForestClassifier(class_weight='balanced', random_state=seed)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_score = model.predict_proba(X_test)[:,1]
print('Random Forest Classifier:')
print(classification_report(y_pred, y_test, labels=[0,1], target_names=['Non-Fraud [0]', 'Fraud [1]']), '\n')

fig, ax = plt.subplots(1, 2, figsize=(20,5))
ax[0].set_title('Confusion Matrix of Random Forest Model:')
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, colorbar=False, values_format='', cmap='crest', ax=ax[0])
ax[0].grid(False)

fpr, tpr, thresholds = roc_curve(y_test, y_pred_score)
roc_auc = auc(fpr, tpr)                       
ax[1].set_title('ROC Curve - Random Forest Classifier')
ax[1].plot(fpr, tpr, label = 'AUC = %0.3f' % roc_auc, c='steelblue')
ax[1].plot([0,1],[0,1],'--', c='lightsteelblue')
ax[1].legend(loc='lower right')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_xlabel('False Positive Rate')